In [2]:
import pandas as pd
import fastf1
from pathlib import Path

fastf1.Cache.enable_cache('../cache')


In [ ]:
from pathlib import Path
import pandas as pd

ROOT_DIR = Path.cwd().parent.resolve()

year = 2025

raw_folder = ROOT_DIR / "data" / "raw" / str(year)

if not raw_folder.exists():
    raise FileNotFoundError(
        f"Could not find {raw_folder}"
    )

races = [race for race in raw_folder.iterdir() if race.is_dir()]

all_pace_metrics = []

for race in races:

    race_name = race.name

    results_file = race / "results.csv"
    laps_file = race / "laps.csv"

    if not results_file.exists() or not laps_file.exists():
        continue

    results_data = pd.read_csv(results_file)
    laps_data = pd.read_csv(laps_file)

    laps_data = laps_data[laps_data["LapTime"].notna()].copy()

    if "TrackStatus" in laps_data.columns:
        laps_data = laps_data[
            laps_data["TrackStatus"].astype(str) == "1"
        ]

    if laps_data.empty:
        continue

    laps_data["LapTime"] = pd.to_timedelta(
        laps_data["LapTime"]
    ).dt.total_seconds()

    race_median = laps_data["LapTime"].median()
    race_fastest = laps_data["LapTime"].min()

    for driver in laps_data["Driver"].unique():

        driver_laps = laps_data[
            laps_data["Driver"] == driver
        ]

        if len(driver_laps) < 5:
            continue

        representative_pace = driver_laps["LapTime"].median()
        fastest_lap = driver_laps["LapTime"].min()
        pace_std = driver_laps["LapTime"].std()

        relative_pace = representative_pace / race_median
        fastest_lap_delta = fastest_lap - race_fastest

        all_pace_metrics.append(
            {
                "Race": race_name,
                "Driver": driver,
                "RepresentativePace": representative_pace,
                "RelativePace": relative_pace,
                "FastestLapDelta": fastest_lap_delta,
                "PaceStdDev": pace_std
            }
        )

driver_race_metrics = pd.DataFrame(all_pace_metrics)

driver_race_metrics.head()

,Race,Driver,RepresentativePace,RelativePace,FastestLapDelta,PaceStdDev
0,abu_dhabi,VER,88.4640,0.985897,0.900,2.801134
1,abu_dhabi,PIA,89.2800,0.994990,0.040,2.810374
2,abu_dhabi,NOR,88.4045,0.985233,0.093,4.021386
3,abu_dhabi,LEC,88.6220,0.987657,0.000,4.000870
4,abu_dhabi,RUS,89.4020,0.996350,1.874,2.791247
